# Vector Hardware Routes

Ideal radial/azimuthal vector Bessel beams remain useful reference targets, but their hardware status is future_hardware_required or simulation_only under the current bench assumptions.


In [1]:
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import bessel_twin_core as bt
from vbb_study import setup_study, vbb_vector, vbb_style
from vbb_study.publication import vector as vector_schema

PATHS = setup_study.bootstrap(Path.cwd())
PRESET = "fast"
RUN_ID = PATHS.get("run_id") or None
out_csv = PATHS["csv"] / "vector"
out_fig = PATHS["figures"] / "vector"
compat_csv = PATHS["csv"] / "publication_study"
out_csv.mkdir(parents=True, exist_ok=True)
out_fig.mkdir(parents=True, exist_ok=True)
compat_csv.mkdir(parents=True, exist_ok=True)
vbb_style.apply_style()

cfg = bt.default_config(PRESET)
design = bt.compute_design_from_targets(cfg.laser, cfg.target, cfg.material)
grid = bt.make_xy_grid(256, 0.18 * bt.um)
KR = 0.95 / bt.um
WAIST = 48.0 * bt.um
ELL_VALUES = (1, 3)


## Editable Notebook Controls

<!-- STAGE88: editable controls -->

This cell exposes the intended user-editable controls for exploratory runs. The locked stage logic below is preserved: changing these controls is for local investigation unless the notebook explicitly wires a value into a regenerated canonical output. Keep QA, caveats, and fail/marginal labels visible. For fast beam-to-sample exploration use the quicklook notebook; for publication-grade outputs use the locked stage runner.


In [ ]:
# STAGE88: visible editable controls for exploratory notebook use.
# Edit NOTEBOOK_CONTROLS below and re-run this cell to apply parameter
# overrides to `cfg` before running any study cell below.
from vbb_study.publication import notebook_controls as nb_controls
from vbb_study.config import um as _um

NOTEBOOK_CONTROLS = nb_controls.make_notebook_controls(
    stage='vector',
    # ── edit these to override the base configuration ───────────────────────
    ell=1,
    target_core_diameter_um=3.0,
    target_bessel_length_um=150.0,
    objective_NA=0.45,
)

# Wire control parameters into `cfg` so downstream cells use them.
_p = NOTEBOOK_CONTROLS.parameters or {}
if "ell" in _p:
    cfg = replace(cfg, target=replace(cfg.target, ell=int(_p["ell"])))
if "target_core_diameter_um" in _p:
    cfg = replace(cfg, target=replace(cfg.target, target_core_diameter_m=float(_p["target_core_diameter_um"]) * _um))
if "target_bessel_length_um" in _p:
    cfg = replace(cfg, target=replace(cfg.target, target_bessel_length_m=float(_p["target_bessel_length_um"]) * _um))
if "objective_NA" in _p:
    cfg = replace(cfg, objective=replace(cfg.objective, NA=float(_p["objective_NA"])))

try:
    display(nb_controls.describe_controls(NOTEBOOK_CONTROLS))
except NameError:
    print(nb_controls.describe_controls(NOTEBOOK_CONTROLS).to_string(index=False))


In [ ]:
# Interactive quicklook — adjust sliders and click "Update plots".
# Runs a fast preview only; nothing is saved.
from vbb_study.publication import notebook_widgets as nbw

_panel = nbw.interactive_quicklook(cfg, method='holographic', preset='fast', ell_range=(0, 6))
display(_panel)


In [2]:
def _stamp(row):
    vector_schema.annotate_vector_row(row, run_id=RUN_ID, qa_status="route_catalogued")
    return row

routes = [
    {
        "case_id": "current_lab_case1_sop",
        "path": "vector_hardware_routes",
        "beam_family": "vector",
        "model_level": "current_lab_approximation",
        "generation_method": "two_slm_same_axis_sop",
        "vector_mode": "sop_encoded_case1",
        "vector_model": "current_lab_case1_sop_encoded",
        "vector_program": "H_shaped_plus_V_reference",
        "vector_method": "method_A_or_B",
        "vector_encoder_hardware": "case1_same_axis_no_waveplates",
        "lab_realizable": True,
        "simulation_only": False,
        "requires_element": "none",
        "uses_waveplates": False,
        "uses_two_slm": True,
        "uses_shared_director_axis": True,
        "route_note": "Limited SOP-encoded approximation only; not true radial/azimuthal vector generation.",
    },
    {
        "case_id": "ideal_radial_current_bench_request",
        "path": "vector_hardware_routes",
        "beam_family": "vector",
        "model_level": "future_hardware_route",
        "generation_method": "qplate_or_vector_converter",
        "vector_mode": "radial",
        "vector_model": "future_true_vector_route",
        "vector_program": "true_radial_target",
        "vector_method": "requires_extra_hardware",
        "vector_encoder_hardware": "not_current_bench",
        "lab_realizable": False,
        "simulation_only": False,
        "requires_element": "qplate_or_vector_mode_converter",
        "uses_waveplates": False,
        "uses_two_slm": False,
        "uses_shared_director_axis": False,
        "route_note": "Ideal radial target is useful, but current Case 1 does not implement it.",
    },
    {
        "case_id": "qplate_or_vector_converter_route",
        "path": "vector_hardware_routes",
        "beam_family": "vector",
        "model_level": "future_hardware_route",
        "generation_method": "qplate_or_vector_converter",
        "vector_mode": "radial",
        "vector_model": "future_true_vector_route",
        "vector_program": "polarisation_converter",
        "vector_method": "future_qplate_or_converter",
        "vector_encoder_hardware": "qplate_or_vector_mode_converter",
        "lab_realizable": False,
        "simulation_only": False,
        "requires_element": "qplate_or_vector_mode_converter",
        "uses_waveplates": False,
        "uses_two_slm": True,
        "uses_shared_director_axis": False,
        "route_note": "Future route for true vector modes if added and aligned.",
    },
    {
        "case_id": "interferometric_combiner_route",
        "path": "vector_hardware_routes",
        "beam_family": "vector",
        "model_level": "future_hardware_route",
        "generation_method": "interferometric_vector_combiner",
        "vector_mode": "hybrid",
        "vector_model": "future_true_vector_route",
        "vector_program": "independent_axes_recombined",
        "vector_method": "periscope_sagnac_or_common_path",
        "vector_encoder_hardware": "interferometric_combiner",
        "lab_realizable": False,
        "simulation_only": False,
        "requires_element": "interferometric_combiner",
        "uses_waveplates": True,
        "uses_two_slm": True,
        "uses_shared_director_axis": False,
        "route_note": "Requires independent polarisation-axis modulation and stable recombination.",
    },
    {
        "case_id": "paper_replica_baliyan_nishchal",
        "path": "vector_hardware_routes",
        "beam_family": "vector",
        "model_level": "paper_replica",
        "generation_method": "paper_replica_simulation",
        "vector_mode": "paper_replica",
        "vector_model": "paper_replica_baliyan_nishchal",
        "vector_program": "paper_jones_chain",
        "vector_method": "diagnostic_benchmark",
        "vector_encoder_hardware": "paper_qwp_hwp_chain",
        "lab_realizable": False,
        "simulation_only": True,
        "requires_element": "waveplate_chain",
        "uses_waveplates": True,
        "uses_two_slm": True,
        "uses_shared_director_axis": False,
        "route_note": "Simulation-only paper-replica diagnostic under current bench assumptions.",
    },
]

summary = vector_schema.ordered_vector_frame(_stamp(dict(row)) for row in routes)
summary.to_csv(out_csv / "vector_hardware_routes_summary.csv", index=False)

readme = PATHS["csv"] / "README.md"
readme.write_text(
    "# CSV output naming\n\n"
    "Stage 5 vector notebooks write canonical outputs under `outputs/csv/vector/`.\n"
    "Compatibility copies for older publication-export names are kept under `outputs/csv/publication_study/` where useful.\n\n"
    "| Old name | Canonical Stage 5 name |\n"
    "| --- | --- |\n"
    "| `vector_atlas_scalar_sas_summary.csv` | `vector/vector_atlas_scalar_sas_summary.csv` and `vector/vector_beam_theory_atlas.csv` |\n"
    "| `vector_atlas_jones_summary.csv` | `vector/vector_atlas_jones_summary.csv` and `vector/vector_beam_theory_atlas.csv` |\n"
    "| `stage6_fidelity_ladder_summary.csv` | `vector/vector_ideal_vs_lab_case1_summary.csv` |\n"
    "| `stage6_fidelity_delta_table.csv` | `vector/stage6_fidelity_delta_table.csv` compatibility diagnostic |\n"
    "| `stage6_slm_encoded_vector_summary.csv` | `vector/vector_ideal_vs_lab_case1_summary.csv` filtered to current Case 1 |\n"
    "| `stage6_paper_replica_vector_summary.csv` | `vector/vector_ideal_vs_lab_case1_summary.csv` filtered to paper-replica diagnostics |\n",
    encoding="utf-8",
)
summary


,run_id,generated_at_utc,source_schema_version,case_id,preset,path,beam_family,model_level,generation_method,hardware_status,...,vector_encoder_hardware,lab_realizable,simulation_only,requires_element,uses_waveplates,uses_two_slm,uses_shared_director_axis,encoded_power_fraction,scalar_reference_case_id,route_note
0,20260604T150311Z,2026-06-04T15:04:39.673554+00:00,1.0.0,current_lab_case1_sop,fast,vector_hardware_routes,vector,current_lab_approximation,two_slm_same_axis_sop,current_lab_realizable,...,case1_same_axis_no_waveplates,True,False,none,False,True,True,<NA>,<NA>,Limited SOP-encoded approximation only; not tr...
1,20260604T150311Z,2026-06-04T15:04:39.673682+00:00,1.0.0,ideal_radial_current_bench_request,fast,vector_hardware_routes,vector,future_hardware_route,qplate_or_vector_converter,future_hardware_required,...,not_current_bench,False,False,qplate_or_vector_mode_converter,False,False,False,<NA>,<NA>,"Ideal radial target is useful, but current Cas..."
2,20260604T150311Z,2026-06-04T15:04:39.673765+00:00,1.0.0,qplate_or_vector_converter_route,fast,vector_hardware_routes,vector,future_hardware_route,qplate_or_vector_converter,future_hardware_required,...,qplate_or_vector_mode_converter,False,False,qplate_or_vector_mode_converter,False,True,False,<NA>,<NA>,Future route for true vector modes if added an...
3,20260604T150311Z,2026-06-04T15:04:39.673828+00:00,1.0.0,interferometric_combiner_route,fast,vector_hardware_routes,vector,future_hardware_route,interferometric_vector_combiner,future_hardware_required,...,interferometric_combiner,False,False,interferometric_combiner,True,True,False,<NA>,<NA>,Requires independent polarisation-axis modulat...
4,20260604T150311Z,2026-06-04T15:04:39.673880+00:00,1.0.0,paper_replica_baliyan_nishchal,fast,vector_hardware_routes,vector,paper_replica,paper_replica_simulation,simulation_only,...,paper_qwp_hwp_chain,False,True,waveplate_chain,True,True,False,<NA>,<NA>,Simulation-only paper-replica diagnostic under...
